# PROJET NEOVOLT - DETECTION DE FRAUDE
## CPDIA Data Scientist

Ce notebook reprend et explique un pipeline complet de détection de fraude sur des données de consommation électrique.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression


## Chargement des données

In [2]:
DATA_PATH = Path("donnees")

consommation = pd.read_csv(DATA_PATH / "releves_consommation.csv")
compteurs = pd.read_csv(DATA_PATH / "compteurs.csv")
clients = pd.read_csv(DATA_PATH / "clients.csv")
fraudes = pd.read_csv(DATA_PATH / "cas_fraude_confirmes.csv")

print("Fichiers chargés")

Fichiers chargés


## Conversion des dates

In [3]:
consommation["date"] = pd.to_datetime(consommation["date"], errors="coerce")
clients["date_entree"] = pd.to_datetime(clients["date_entree"], errors="coerce")
compteurs["date_pose"] = pd.to_datetime(compteurs["date_pose"], errors="coerce")
fraudes["date_detection"] = pd.to_datetime(fraudes["date_detection"], errors="coerce")

## Nettoyage des données

In [ ]:
consommation = consommation.drop_duplicates()

consommation["consommation_kwh"] = pd.to_numeric(consommation["consommation_kwh"], errors="coerce")

consommation = consommation[
    consommation["consommation_kwh"].isna()
    | (consommation["consommation_kwh"] >= 0)
]

consommation["consommation_kwh"] = consommation["consommation_kwh"].interpolate()
consommation = consommation.dropna(subset=["consommation_kwh"])

clients["nb_personnes_foyer"] = clients["nb_personnes_foyer"].fillna(0)
clients["surface_m2"] = clients["surface_m2"].fillna(clients["surface_m2"].median())

compteurs = compteurs[compteurs["statut"] == "actif"]

## Fusion des données

In [5]:
if "zone" in compteurs.columns:
    compteurs = compteurs.drop(columns=["zone"])

df = consommation.merge(compteurs, on="id_pdl", how="left")
df = df.merge(clients, on="id_client", how="left")

print(df.shape)

(510668, 17)


## Feature Engineering

In [6]:
df = df.sort_values(["id_pdl", "date"])

df["conso_j_1"] = df.groupby("id_pdl")["consommation_kwh"].shift(1)

df["moyenne_7j"] = df.groupby("id_pdl")["consommation_kwh"].transform(
    lambda x: x.rolling(7, min_periods=1).mean()
)

df["std_7j"] = df.groupby("id_pdl")["consommation_kwh"].transform(
    lambda x: x.rolling(7, min_periods=1).std()
)

df["variation_pct"] = df.groupby("id_pdl")["consommation_kwh"].pct_change()

df["conso_par_m2"] = df["consommation_kwh"] / df["surface_m2"].replace(0, np.nan)

df["jour_semaine"] = df["date"].dt.dayofweek
df["mois"] = df["date"].dt.month
df["weekend"] = (df["jour_semaine"] >= 5).astype(int)

df.replace([np.inf, -np.inf], np.nan, inplace=True)


## Création du label fraude

In [7]:
fraudes_ids = set(fraudes["id_pdl"].unique())

df["fraude"] = df["id_pdl"].isin(fraudes_ids).astype(int)

print(df["fraude"].value_counts())

fraude
0    493151
1     17517
Name: count, dtype: int64
